# **6. GÜN : Train / Validation / Test**
* Model geliştirme sürecine başlıyorum. Algoritmaları eğitmeye geçmeden önce elimdeki temizlenmiş veri setimi istenilen oranlara uygun şekilde bölüyorum. Sklearn kütüphanesini kullanarak verinin %70 kısmını train, %15 kısmını validation ve %15 kısmını test olacak şekilde ayırdım.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/eco_driving_score_cleaned.csv')

X = df.drop('fuel_consumption', axis=1)
y = df['fuel_consumption']

# Önce verinin %70 kısmını train, kalan %30 kısmını geçici bir sete ayırdım
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Kalan %30 kısmı ikiye bölerek %15 validation ve %15 test setlerini oluşturdum
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print("Train seti boyutu:", len(X_train))
print("Validation seti boyutu:", len(X_val))
print("Test seti boyutu:", len(X_test))

Train seti boyutu: 20493
Validation seti boyutu: 4391
Test seti boyutu: 4392


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('/content/drive/MyDrive/eco_driving_score_cleaned.csv')

## ***Araştırdığım Kavramlar :***

* Train set: Modeli doğrudan eğittiğim ana veri setidir.  Validation set: Eğitim aşamasında modelin performansını ölçtüğüm ve parametre ayarlarını yaptığım veri setidir.

* Test set: Modelin daha önce hiç görmediği ve proje sonundaki nihai başarısını test ettiğim veri setidir.  

* Overfitting: Modelin eğitim verisini ezberleyerek aşırı öğrenmesi ve yeni verilerde başarısız olması durumudur.

* Data leakage: Modelin eğitim aşamasında test verisinden kopya çekmesi ve hatalı şekilde yüksek başarı göstermesidir.

## GÜN SONU SORUSU : Model seçimi sırasında test dataset neden kullanılmamalıdır?

* Test setini model seçimi için kullanırsam model bu veriye aşina olur. Bu durumda test seti tarafsızlığını kaybeder ve modelin gerçek dünyadaki başarısı hakkında beni yanıltır. performans ölçümü için test verisinin tamamen izole kalması şarttır.

# **7. GÜN: Baseline Model**

Veri setimi eğitim ve doğrulama olarak böldükten sonra, projenin referans noktasını oluşturacak ilk temel modelimi eğitiyorum. Karmaşık algoritmalara geçmeden önce lineer Regresyon algoritmasını kullanarak sistemin en temel performansını ölçeceğim.

Gerekli kütüphaneleri çağırıp modelimi X_train ve y_train üzerinde eğitiyorum. Ardından X_val setini kullanarak tahminler üretiyor ve hata metriklerini hesaplıyorum.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Modeli tanımlıyorum ve eğitiyorum
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Validation seti üzerinde tahmin yapıyorum
y_pred_val = baseline_model.predict(X_val)

# Hata metriklerini hesaplıyorum
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

print("Temel Model Performansı:")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Skoru: {r2:.4f}")

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

#Modeli tanımlıyorum ve eğitime başlıoyuem
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

#Validation seti tahmini...
y_pred_val = baseline_model.predict(X_val)

#Hata metriklerini hesaplama
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)

print("bASE MODEL Performansı:")
print(f"MSE: {mse: 4f}")
print(f"RAMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2 Skoru: {r2:.4f}")

bASE MODEL Performansı:
MSE:  1.416975
RAMSE: 1.1904
MAE: 0.9564
R2 Skoru: 0.3266


## **SONUÇLAR**
* R2 Skoru 0.3266 çıkarak yakıt tüketimindeki değişimlerin sadece yüzde 32 kadarını açıklayabildiğini gösteriyor. Hedefim bu oranı yüzde 70 ve üzerine çıkarmak olacak.

* MAE değeri 0.9564 seviyesinde. Tahminlerimde ortalama 1 litreye yakın sapma yapıyorum. Ortalama tüketimin 8 litre civarında olduğunu düşünürsek, modelin yaklaşık yüzde 12 oranında bir hata payı var.

* RMSE değeri 1.1904 çıkarak tahminlerimde büyük hataların da yer aldığını doğruluyor.

### **NEDEN DÜŞÜK ÇIKTI**

* Doğrusal regresyon algoritması sürüş verileri arasındaki karmaşık ilişkileri yakalamakta oldukça yetersiz. Veriler üzerinde herhangi bir ölçeklendirme işlemi yapmadığım için rölanti süresi gibi büyük sayılara sahip sütunlar modeli ciddi şekilde yanılttı.